# Part I - PyTorch Training Pipeline - Churn Modelling Example

Goal: Based upon data of customers of a bank we try to predict whether a customer plans to stay or leave the company.

Aim: To understand how neural networks work by manually constructing the model


**Work Flow**

Step 1. Load dataset

Step 2. Basic preprocessing

Step 3. Training process:

<div align='center'>

Create model

$\downarrow$

Forward pass

$\downarrow$

Loss computation

$\downarrow$

Backward pass

$\downarrow$

Update weights

</div>

Step 4. Model prediction and evaluation

## 1. Load Dataset

In [32]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("shubh0799/churn-modelling")

print("Path to dataset files:", path)
print("Files:", os.listdir(path))

Using Colab cache for faster access to the 'churn-modelling' dataset.
Path to dataset files: /kaggle/input/churn-modelling
Files: ['Churn_Modelling.csv']


In [33]:
import pandas as pd
import numpy as np
import torch
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split


In [34]:
data = pd.read_csv(os.path.join(path, "Churn_Modelling.csv"))
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 2. Basic preprocessing

* Remove RowNumber, CustomerId, Surname

In [35]:
df = data.copy()
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


* Rename column names in snakecase convention

In [36]:
df.columns = df.columns.str.replace(r'(?<!^)(?=[A-Z])', '_', regex=True).str.lower()
df.head()

,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


* Gather basic information

In [37]:
df.shape

(10000, 11)

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   credit_score      10000 non-null  int64  
 1   geography         10000 non-null  object 
 2   gender            10000 non-null  object 
 3   age               10000 non-null  int64  
 4   tenure            10000 non-null  int64  
 5   balance           10000 non-null  float64
 6   num_of_products   10000 non-null  int64  
 7   has_cr_card       10000 non-null  int64  
 8   is_active_member  10000 non-null  int64  
 9   estimated_salary  10000 non-null  float64
 10  exited            10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


No missing values are found.

**Numerical columns:**
* credit_score
* age
* tenure
* balance
* num_of_products
* estimated_salary

**Categorical columns:**
* geography
* gender
* has_cr_card
* is_active_member
* exited (target)

In [39]:
df.describe()

,credit_score,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


* Remove duplicate records (if any)

In [40]:
df.duplicated().sum()

np.int64(0)

* Split the data into train and test

In [41]:
X = df.drop(columns=['exited'])
y = df['exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

* Scale the numerical columns
* One-hot encode the categorical columns (ignore 'has_cr_card' and 'is_active_member')
* Label encode the target column (not necessary in this case)

In [42]:
num_cols = ['credit_score', 'age', 'tenure', 'balance', 'num_of_products', 'estimated_salary']
cat_cols = ['geography', 'gender']
passthrough_cols = ['has_cr_card', 'is_active_member']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
        ('pass', 'passthrough', passthrough_cols)
    ],
    verbose_feature_names_out=False
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

In [43]:
X_train_proc

array([[ 0.35649971, -0.6557859 ,  0.34567966, ...,  1.        ,
         1.        ,  1.        ],
       [-0.20389777,  0.29493847, -0.3483691 , ...,  1.        ,
         1.        ,  1.        ],
       [-0.96147213, -1.41636539, -0.69539349, ...,  1.        ,
         1.        ,  0.        ],
       ...,
       [ 0.86500853, -0.08535128, -1.38944225, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.15932282,  0.3900109 ,  1.03972843, ...,  1.        ,
         1.        ,  0.        ],
       [ 0.47065475,  1.15059039, -1.38944225, ...,  1.        ,
         1.        ,  1.        ]])

In [44]:
X_test_proc

array([[-0.57749609, -0.6557859 , -0.69539349, ...,  1.        ,
         0.        ,  0.        ],
       [-0.29729735,  0.3900109 , -1.38944225, ...,  1.        ,
         1.        ,  1.        ],
       [-0.52560743,  0.48508334, -0.3483691 , ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [ 0.81311987,  0.77030065,  0.69270405, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.41876609, -0.94100321, -0.3483691 , ...,  1.        ,
         1.        ,  0.        ],
       [-0.24540869,  0.00972116, -1.38944225, ...,  1.        ,
         1.        ,  1.        ]])

* Convert from numpy arrays to torch tensors

In [45]:
X_train_tensor = torch.from_numpy(X_train_proc)
X_test_tensor = torch.from_numpy(X_test_proc)
y_train_tensor = torch.from_numpy(y_train.values)
y_test_tensor = torch.from_numpy(y_test.values)

In [46]:
print(X_train_tensor.shape, X_train_tensor.dtype)
print(y_train_tensor.shape, y_train_tensor.dtype)
print(X_test_tensor.shape, X_test_tensor.dtype)
print(y_test_tensor.shape, y_test_tensor.dtype)

torch.Size([8000, 11]) torch.float64
torch.Size([8000]) torch.int64
torch.Size([2000, 11]) torch.float64
torch.Size([2000]) torch.int64


**Note:** By default PyTorch gives preference to float32 since it takes up less memory and does faster computation on GPU. Thus it is better to convert the data types of our dataset to float32 instead of retaining them as 64-bit numbers.

In [47]:
X_train_tensor = X_train_tensor.to(torch.float32)
X_test_tensor = X_test_tensor.to(torch.float32)
y_train_tensor = y_train_tensor.to(torch.float32)
y_test_tensor = y_test_tensor.to(torch.float32)

print(X_train_tensor.shape, X_train_tensor.dtype)
print(y_train_tensor.shape, y_train_tensor.dtype)
print(X_test_tensor.shape, X_test_tensor.dtype)
print(y_test_tensor.shape, y_test_tensor.dtype)

torch.Size([8000, 11]) torch.float32
torch.Size([8000]) torch.float32
torch.Size([2000, 11]) torch.float32
torch.Size([2000]) torch.float32


## 3. Construct the neural network and create model instance

* Create a neural network object containing forward pass and loss computation functionalities

In [48]:
class SimpleNeuralNetwork():
  def __init__(self, X_training):
    self.weights = torch.rand(X_training.shape[1], 1, requires_grad=True)
    self.bias = torch.zeros(1, requires_grad=True)

  def forward_pass(self, X_training):
    z = torch.matmul(X_training, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_comp(self, y_pred, y_actual):
    epsilon = 1e-8
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
    loss = - (y_actual*torch.log(y_pred) + (1-y_actual)*torch.log(1-y_pred)).mean()
    return loss

* Create model instance

In [49]:
model = SimpleNeuralNetwork(X_train_tensor)

## 4. Training process

In [50]:
epochs = 10 # adjust the number of epochs to get the least amount of loss
alpha = 0.01 # learning rate (also adjustable)

In [51]:
for epoch in range(epochs):

  # forward pass
  y_pred = model.forward_pass(X_train_tensor)

  # loss
  loss = model.loss_comp(y_pred, y_train_tensor)
  #print(f"Epoch: {epoch+1}, Loss: {loss.item()}") --> to see how loss decreases

  # backward pass
  loss.backward()

  # update weights and bias
  with torch.no_grad():
    model.weights -= alpha * model.weights.grad
    model.bias -= alpha * model.bias.grad

  # set current gradient to zero to avoid accumulation in the next epoch
  model.weights.grad.zero_()
  model.bias.grad.zero_()

In [52]:
loss.item() # final loss

1.3106440305709839

## 5. Model prediction and evaluation

In [53]:
with torch.no_grad():
  y_fpred = model.forward_pass(X_test_tensor)
  threshold = 0.8 # adjustable
  for i in range(len(y_fpred)):
    if y_fpred[i] < threshold:
      y_fpred[i] = 0
    else:
      y_fpred[i] = 1

y_fpred

tensor([[0.],
        [0.],
        [1.],
        ...,
        [1.],
        [0.],
        [0.]])

In [54]:
accuracy = (y_fpred == y_test_tensor).float().mean()
print(f"Accuracy: {accuracy.item()}")

Accuracy: 0.5616105198860168


# Part II - PyTorch Training Pipeline - Churn Modelling Example (Improvised)

Goal: Based upon data of customers of a bank we try to predict whether a customer plans to stay or leave the company.

Aim: To build a neural network model using nn library.

**Work Flow**

Step 1. Load dataset

Step 2. Basic preprocessing

Step 3. Training process:

<div align='center'>

Create model

$\downarrow$

Forward pass

$\downarrow$

Loss computation

$\downarrow$

Backward pass

$\downarrow$

Update weights

</div>

Step 4. Model prediction and evaluation

**Note:** Run all the cells in section 2 before proceeding the rest of this part.

## 1. Construct the neural network using nn.Module

The nn.Module is the fundamental class for all neural network building blocks. We construct our neural network class by inheriting from this class.

In [55]:
import torch.nn as nn

class SimpleNNModel(nn.Module):
  def __init__(self, num_features):

    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(num_features, 5),
        nn.ReLU(),
        nn.Linear(5, 1),
        nn.Sigmoid()
    )

  def forward(self, features):
    out = self.network(features)
    return out

1. We initialize our neural network class by inheriting the init function of `nn.Module`.
2. Then we specify the structure of our network. A neural network has layers of neurons. We specify the layers sequentially or in other words, in order. `nn.Sequential` is a container that holds all the layers together.
3. The layer computes mathematical operations such as applying linear transformation, bilinear transformation, etc. to the input data. We are using linear transformation which results in performing the below mathematical operation.

<div align='center'>

$y = x*w + b$

</div>

4. More specifically, we have one input layer, one hidden layer, and one output layer. For the hidden layer, we are using ReLU activation function and for the output layer, we are using Sigmoid function that gives output between 0 and 1. This concludes the structure of our neural network.
5. Functions of a neural network include forward and backward propagation. While forward propagation is explicitly mentioned, backward pass is not defined because the module handles backward track automatically.
6. In the forward pass, we obtain the output by simply calling out the network while passing our training dataset.

## 2. Create model instance and mention the preferred loss function and optimizer

In [56]:
my_model = SimpleNNModel(X_train_tensor.shape[1])

loss_func = nn.BCELoss()

optimizer = torch.optim.SGD(my_model.parameters(), lr=0.01)
# optimizer = torch.optim.Adam(my_model.parameters(), lr=0.01)

## 3. Training process

In [57]:
epochs = 10

for epoch in range(epochs):

  y_pred_nn = my_model(X_train_tensor)                    # forward pass
  loss_ = loss_func(y_pred_nn, y_train_tensor.view(-1,1)) # loss computation
  optimizer.zero_grad()                                   # clear gradients
  loss_.backward()                                        # backward pass
  optimizer.step()                                        # update weights
  print(f'Epoch: {epoch + 1}, Loss: {loss_.item()}')

Epoch: 1, Loss: 0.5920566916465759
Epoch: 2, Loss: 0.5914366245269775
Epoch: 3, Loss: 0.5908203721046448
Epoch: 4, Loss: 0.590208113193512
Epoch: 5, Loss: 0.589600145816803
Epoch: 6, Loss: 0.5889961123466492
Epoch: 7, Loss: 0.588395893573761
Epoch: 8, Loss: 0.5877997279167175
Epoch: 9, Loss: 0.587207555770874
Epoch: 10, Loss: 0.5866199135780334


## 4. Get model summary

Pip install torchinfo library and import summary function

In [58]:
!pip install torchinfo # run this cell if torchinfo is not installed in your local

In [59]:
from torchinfo import summary
summary(my_model, input_size=(8000, 11))

Layer (type:depth-idx)                   Output Shape              Param #
SimpleNNModel                            [8000, 1]                 --
├─Sequential: 1-1                        [8000, 1]                 --
│    └─Linear: 2-1                       [8000, 5]                 60
│    └─ReLU: 2-2                         [8000, 5]                 --
│    └─Linear: 2-3                       [8000, 1]                 6
│    └─Sigmoid: 2-4                      [8000, 1]                 --
Total params: 66
Trainable params: 66
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.53
Input size (MB): 0.35
Forward/backward pass size (MB): 0.38
Params size (MB): 0.00
Estimated Total Size (MB): 0.74

Number of parameters corresponding to the first layer is 60 because we pass 11 features to 5 neurons, resulting in 55 weights and 5 biases (one bias for each neuron). Likewise for the second layer, the output of the first layer has 5 neurons, which act as the 5 inputs of the second layer, fed into the final layer containing one neuron. Therefore, there will be 5 weights and one bias, accounting for a total of 6 parameters.

## 5. Model predictions and evaluation

In [60]:
with torch.no_grad():
  y_pred_t = my_model(X_test_tensor)
  y_pred_t = (y_pred_t > 0.8).float()
  accuracy = (y_pred_t == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.8034999966621399
